## 1. Initialize Project Environment
Import libraries for tree visualization.

In [1]:
from __future__ import annotations

import logging
from dataclasses import dataclass, asdict
from pathlib import Path
from typing import Dict

import matplotlib

matplotlib.use("Agg")  # Non-interactive backend for saving figures
import matplotlib.pyplot as plt
from Bio import Phylo

logging.basicConfig(level=logging.INFO, format="[%(levelname)s] %(message)s")

## 2. Define Configuration Parameters

In [2]:
@dataclass
class VizConfig:
    tree_path: Path = Path("artifacts/task2_nj_tree.nwk")
    export_dir: Path = Path("artifacts")
    figure_size: tuple = (14, 10)
    dpi: int = 150

    def describe(self) -> Dict[str, str]:
        info = asdict(self)
        info["tree_path"] = str(info["tree_path"])
        info["export_dir"] = str(info["export_dir"])
        return info


CONFIG = VizConfig()
CONFIG.describe()

{'tree_path': 'artifacts/task2_nj_tree.nwk',
 'export_dir': 'artifacts',
 'figure_size': (14, 10),
 'dpi': 150}

## 3. Load and Visualize Tree

In [3]:
def load_tree(path: Path):
    """Load tree from Newick file."""
    if not path.exists():
        raise FileNotFoundError(f"Tree not found: {path}. Run Task2 first.")

    tree = Phylo.read(path, "newick")
    logging.info("Loaded tree from %s", path)
    return tree


tree = load_tree(CONFIG.tree_path)
print("Tree structure:")
Phylo.draw_ascii(tree)

[INFO] Loaded tree from artifacts/task2_nj_tree.nwk


Tree structure:
                           ___________________________________ XM_005194938.2
           _______________|
  ________|               |_____________________________ NM_001317019.1
 |        |
 |        |_____________________ XM_006719566.3
 |
 |_______________ NM_000546.6
 |
 |  _____________ NM_131327.2
 | |
 | |        ____________ XM_031279688.1
_|_|     __|
 | |    |  |________ NM_001006919.1
 | |    |
 | |____|___________ NM_001085860.1
 |      |
 |      |__________ NM_011640.3
 |
 |______________ NM_001089263.1



In [4]:
# Species name mapping for better labels
SPECIES_NAMES = {
    "NM_000546.6": "Human (H. sapiens)",
    "NM_011640.3": "Mouse (M. musculus)",
    "NM_131327.2": "Zebrafish (D. rerio)",
    "XM_006719566.3": "Dog (C. familiaris)",
    "NM_001317019.1": "Pig (S. scrofa)",
    "NM_001085860.1": "Cattle (B. taurus)",
    "XM_005194938.2": "Dusky titi (C. moloch)",
    "NM_001006919.1": "Duck (A. platyrhynchos)",
    "NM_001089263.1": "Chicken (G. gallus)",
    "XM_031279688.1": "Lion (P. leo)",
}


def rename_tree_labels(tree, name_map: Dict[str, str]):
    """Rename tree terminal labels for better visualization."""
    for clade in tree.find_clades():
        if clade.name and clade.name in name_map:
            clade.name = name_map[clade.name]
    return tree


tree_labeled = rename_tree_labels(tree, SPECIES_NAMES)

In [5]:
def create_tree_visualization(tree, cfg: VizConfig) -> plt.Figure:
    """Create publication-quality tree visualization."""
    fig, ax = plt.subplots(figsize=cfg.figure_size)

    # Draw tree
    Phylo.draw(tree, axes=ax, do_show=False)

    # Customize appearance
    ax.set_title(
        "Neighbor-Joining Phylogenetic Tree of TP53 Sequences\n(10 Vertebrate Species)",
        fontsize=14,
        fontweight="bold",
        pad=20,
    )
    ax.set_xlabel("Evolutionary Distance (p-distance)", fontsize=12)
    ax.set_ylabel("Species", fontsize=12)

    # Add grid
    ax.grid(True, alpha=0.3, linestyle="--")

    plt.tight_layout()
    return fig


fig = create_tree_visualization(tree_labeled, CONFIG)
plt.show()

/tmp/ipykernel_28408/2745405223.py:26: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## 4. Export Visualization

In [6]:
EXPORT_DIR = CONFIG.export_dir
EXPORT_DIR.mkdir(parents=True, exist_ok=True)

# Save PNG
png_path = EXPORT_DIR / "task4_tree_plot.png"
fig.savefig(png_path, dpi=CONFIG.dpi, bbox_inches="tight", facecolor="white")
print(f"[OK] Tree visualization saved to: {png_path.resolve()}")

# Save PDF (vector format)
pdf_path = EXPORT_DIR / "task4_tree_plot.pdf"
fig.savefig(pdf_path, bbox_inches="tight", facecolor="white")
print(f"[OK] Tree visualization (PDF) saved to: {pdf_path.resolve()}")

plt.close(fig)
print(f"\nArtifacts saved to {EXPORT_DIR.resolve()}")

[OK] Tree visualization saved to: /home/rbals/git/daha-bdhb/BDHB-lab/labs/04_phylogenetics/assignments/artifacts/task4_tree_plot.png
[OK] Tree visualization (PDF) saved to: /home/rbals/git/daha-bdhb/BDHB-lab/labs/04_phylogenetics/assignments/artifacts/task4_tree_plot.pdf

Artifacts saved to /home/rbals/git/daha-bdhb/BDHB-lab/labs/04_phylogenetics/assignments/artifacts
